In [0]:
import re
import pyspark.sql.functions as F
from pyspark.sql.functions import col, get_json_object
from delta.tables import DeltaTable

# Data Cleansing

In [0]:
df = spark.read.table("fraud_detection_project.bronze_layer.device_signals")

def StandardizeNames(df):
    l = df.columns
    cols = []
    for c in l:
        temp = re.sub(r'(?<!^)(?=[A-Z])', '_', c).lower()
        for _ in range(5):
            temp = re.sub(r'\b([a-z])_([a-z])\b', r'\1\2', temp)
            temp = re.sub(r'(?<=[a-z])_([a-z])(?=_|$)', r'\1', temp)
        temp = re.sub(r'_+','_', temp)
        temp = temp.lstrip('_')
        cols.append(temp)
    return df.toDF(*cols)
df = StandardizeNames(df)

df = df.withColumnRenamed("deviceid", "device_id")
df.dtypes

In [0]:
# {"latitude": -7.8036652868763845, "longitude": -37.83341249436084, "accuracy_meters": 34.73, "altitude_meters": null, "timestamp": "2026-02-12 00:37:47.655", "region_code": "PE"}

# Extracting device current location
df = df.withColumn('device_latitude',       get_json_object(col('ip_region'), '$.latitude'))
df = df.withColumn('device_longitude',      get_json_object(col('ip_region'), '$.longitude'))
df = df.withColumn('device_acuracy_meters', get_json_object(col('ip_region'), '$.acuracy_meters'))
df = df.withColumn('device_altitude_meters',get_json_object(col('ip_region'), '$.altitude_meters'))
df = df.withColumn('device_timestamp',      get_json_object(col('ip_region'), '$.timestamp'))
df = df.withColumn('device_region_code',    get_json_object(col('ip_region'), '$.region_code'))

# {"device_form_factor": "MOBILE", "device_os": "Android", "device_model": "Samsung Galaxy S23", "device_browser": null, "device_language": "pt-BR", "app_version": "6.2.0"}
# Extracting device signal details
df = df.withColumn('device_form_factor', get_json_object(col('user_agent_string'), '$.device_form_factor'))
df = df.withColumn('device_os',          get_json_object(col('user_agent_string'), '$.device_os'))
df = df.withColumn('device_model',       get_json_object(col('user_agent_string'), '$.device_model'))
df = df.withColumn('device_browser',     get_json_object(col('user_agent_string'), '$.device_browser'))
df = df.withColumn('device_language',    get_json_object(col('user_agent_string'), '$.device_language'))
df = df.withColumn('app_version',        get_json_object(col('user_agent_string'), '$.app_version'))


In [0]:
# Deleting duplicated data
df.dropDuplicates(['device_id'])

# Deleting rows without some features
df = df.dropna(how='any', subset=['device_id','file_path','ingest_datetime'])

# Deleting unnecessary column
df = df.drop('kafka_topic', 'ip_region', 'user_agent_string') 

In [0]:

target = "fraud_detection_project.silver_layer.device_signals"

if spark.catalog.tableExists(target):
    dt = DeltaTable.forName(spark, target)

    dt.alias("t").merge(
        df.alias("s"),
        "t.device_id = s.device_id"
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()
    print("Merge concluded.")
else:
    df.write.format("delta") \
      .option("mergeSchema", "true") \
      .saveAsTable(target)
    print("Tabel created.")

In [0]:
df.count()

In [0]:
%sql
SELECT * FROM fraud_detection_project.silver_layer.device_signals LIMIT(200)